# 04 - Figures

Reads whatever the runs have produced and renders the figures. Works on a CPU
kernel: it only reads JSON.

Figure 1 is the headline claim (retention as the library grows) and figure 2 is
the cost it is bought at (adapter storage against skill count).

## Load results

In [ ]:
import json
from pathlib import Path

from analysis.collect import summary_table, write_collected

RUN_ID = "gpu_main"
METHODS = ["in_context", "seq_no_replay", "isolated", "ours"]

root = Path("runs") / RUN_ID
results_path = root / "results.json"
if not results_path.exists():
    results_path = write_collected(root, METHODS, run_id=RUN_ID)

results = json.loads(results_path.read_text())
print(summary_table(results))

## Figure 1 - forgetting curve

Retention is normalised per skill against its own score when freshly learned,
so the curve is not just tracking which tasks happen to be old. Skills whose
program calls nothing are excluded: they need no recall, so they cannot show
forgetting.

In [ ]:
from analysis.plots import plot_forgetting_curve

order = [m for m in METHODS if m in results["methods"]]
fig = plot_forgetting_curve(results, root / "figures" / "fig1_forgetting_curve.png",
                            order=order)
fig

## Figure 2 - adapter storage

The honest half of the comparison with PEAM-style isolation: isolation does not
forget, and it pays for that linearly in stored parameters.

In [ ]:
from analysis.plots import plot_parameter_growth

fig = plot_parameter_growth(results, root / "figures" / "fig2_parameter_growth.png",
                            order=order)
fig

## Figure 3 - in-context cost

In [ ]:
from analysis.plots import plot_token_cost

fig = plot_token_cost(results, root / "figures" / "fig3_token_cost.png", order=order)
fig

## Table

In [ ]:
# The numbers as they go into the paper.
rows = []
for key in order:
    m = results["methods"][key]
    rows.append({
        "method": m["label"],
        "final success": round(m["final_success_rate"], 3),
        "oldest 3": round(m["final_earliest_3"], 3),
        "prompt tokens": round(m["tokens"].get("mean_prompt_tokens", 0)),
        "stored params": m["final_stored_params"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)